# Generate predictions from black-box models

### Generating Yb's

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, cross_val_predict
from pandas.io.parsers import read_csv
import pandas as pd

# Might not need this if you don't have GPU
import cupy

In [ ]:
path = './datasets/census/'
# path = './datasets/coupon/'
# path = './datasets/stop&frisk/'

In [ ]:
dftrain = read_csv(path + 'train.csv',header = 0, sep = ',')
dftest = read_csv(path + 'test.csv',header = 0, sep = ',')

# Sync binarized columns as holland only appears in test dataset
trainnum = len(dftrain)
df = pd.concat([dftrain, dftest])
dftrain = df[:trainnum]
dftest = df[trainnum:]

Xtrain = dftrain.drop(columns=["Y"]).to_numpy()
Ytrain = dftrain["Y"].to_numpy()
Xtest = dftest.drop(columns=["Y"]).to_numpy()
Ytest = dftest["Y"].to_numpy()

#### Random Forest

Do grid search on parameters

In [ ]:
param_grid = {
    'n_estimators': [1, 10, 100, 500, 1000],
    'max_features': [1, 10, 100, 1000, 'sqrt', 'log2', None],
}

rf = RandomForestClassifier(random_state=2026)
grid_search = GridSearchCV(rf, param_grid, n_jobs=-1)

In [ ]:
grid_search.fit(Xtrain, Ytrain)

Make predictions with best parameters

In [ ]:
Yhat_test = grid_search.predict(Xtest)
# How is the paper generating Ybtrain without data leakage??
# I have no idea, I'll use cross_val_predict for now
rf_best = RandomForestClassifier(grid_search.best_params_["n_estimators"], max_features=grid_search.best_params_["max_features"], random_state=2026)
Yhat_train = cross_val_predict(rf_best, Xtrain, Ytrain, n_jobs=-1)

In [ ]:
Ybtrain_df = pd.DataFrame({"Yb": Yhat_train})
Ybtrain_df.to_csv(path + 'rf_ybtrain.csv', index=False)
Ybtest_df = pd.DataFrame({"Yb": Yhat_test})
Ybtest_df.to_csv(path + 'rf_ybtest.csv', index=False)

train_accuracy = sum(Yhat_train==Ytrain)/len(Ytrain)
test_accuracy = sum(Yhat_test==Ytest)/len(Ytest)

print(f"Train cross-validation accuracy of rf: {train_accuracy}")
print(f"Test accuracy of rf: {test_accuracy}")

### AdaBoost

Do grid search on parameters

In [ ]:
param_grid = {
    'estimator__max_depth': [1, 3, 5, 10, 30],
    'estimator__max_features': [1, 10, 100, 1000, 'sqrt', 'log2', None],
}

ab = AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=2026), n_estimators=800, random_state=2026)
grid_search = GridSearchCV(ab, param_grid, n_jobs=-1)

In [ ]:
grid_search.fit(Xtrain, Ytrain)

Make predictions with best parameters

In [ ]:
Yhat_test = grid_search.predict(Xtest)
dt_best = DecisionTreeClassifier(
    max_depth=grid_search.best_params_["estimator__max_depth"],
    max_features=grid_search.best_params_["estimator__max_features"],
    random_state=2026
    )
ab_best = AdaBoostClassifier(estimator=dt_best, n_estimators=800, random_state=2026)
Yhat_train = cross_val_predict(ab_best, Xtrain, Ytrain, n_jobs=-1)

In [ ]:
Ybtrain_df = pd.DataFrame({"Yb": Yhat_train})
Ybtrain_df.to_csv(path + 'ab_ybtrain.csv', index=False)
Ybtest_df = pd.DataFrame({"Yb": Yhat_test})
Ybtest_df.to_csv(path + 'ab_ybtest.csv', index=False)

train_accuracy = sum(Yhat_train==Ytrain)/len(Ytrain)
test_accuracy = sum(Yhat_test==Ytest)/len(Ytest)

print(f"Train cross-validation accuracy of ab: {train_accuracy}")
print(f"Test accuracy of ab: {test_accuracy}")

### XGBoost

Do grid search on parameters

In [ ]:
param_grid = {
    'max_depth': [2, 3, 4, 5, 6, 7, 8, 9, 10],
    # Equivalent of "maximum features used by a tree"
    'colsample_bytree': [0.1, 0.2, 0.5, 0.8, 1],
    # No idea how the paper varied "the minimum samples that need to exist in a leaf", min_child_weight seems to be the closest one
    'min_child_weight': [0.01, 0.1, 0.5, 1, 5]
}

# Change device to cpu if needed
xg = XGBClassifier(booster="gbtree", device="cuda", learning_rate=0.1, seed=2026)
grid_search = GridSearchCV(xg, param_grid)

In [ ]:
grid_search.fit(cupy.asarray(Xtrain), Ytrain)

Make predictions with best parameters

In [ ]:
Yhat_test = grid_search.predict(cupy.asarray(Xtest))
xg_best = XGBClassifier(
    booster="gbtree", 
    device="cuda", 
    learning_rate=0.1,
    max_depth=grid_search.best_params_["max_depth"],
    colsample_bytree=grid_search.best_params_["colsample_bytree"],
    min_child_weight=grid_search.best_params_["min_child_weight"],
    random_state=2026
    )
Yhat_train = cross_val_predict(xg_best, cupy.asarray(Xtrain), Ytrain)

In [ ]:
Ybtrain_df = pd.DataFrame({"Yb": Yhat_train})
Ybtrain_df.to_csv(path + 'xg_ybtrain.csv', index=False)
Ybtest_df = pd.DataFrame({"Yb": Yhat_test})
Ybtest_df.to_csv(path + 'xg_ybtest.csv', index=False)

train_accuracy = sum(Yhat_train==Ytrain)/len(Ytrain)
test_accuracy = sum(Yhat_test==Ytest)/len(Ytest)

print(f"Train cross-validation accuracy of xg: {train_accuracy}")
print(f"Test accuracy of xg: {test_accuracy}")